# STFPM v2 — 시간 블록 5-fold, 원본 vs 마스킹 (같은 사진 3,395장)

**왜 하나**
- STFPM(정상 사진만 학습하는 이상 탐지)은 "불량 데이터가 0장일 때"의 기준점이다. 지금까지 결과는 조건이 제각각이라 다른 모델과 같은 조건에서 비교한 적이 없다.
  - 팀원 v2 체크포인트: 5장 간격 분할로 AUROC 0.963. 시험 사진 바로 옆 정상 사진이 학습에 들어간 유리한 조건이다.
  - 예전 코드의 시간 블록 결과: 원본 0.674, 예전 마스킹 + 선별 0.868. 이때는 선별 효과가 섞여 있었다.
- 이번에는 STFPM v2(팀원 10/1 코드, 고치지 않음)를 지금 비교와 **같은 사진 목록, 같은 시간 블록 분할**로 돌린다. 그리고 **입력만 원본 / 마스킹 두 가지로 바꿔서** 마스킹만의 효과를 본다.

| 조건 | 사진 목록 | 입력 |
|---|---|---|
| 원본 | 2단계 선별 + 보류 사진 활용 3,395장 (ConvNeXt 3시드 비교와 같은 목록) | 고정 크롭 1000×1000 그대로 |
| 마스킹 | 같은 3,395장 | 같은 크롭, 마스크 바깥을 회색 (ConvNeXt와 같은 마스크) |

- STFPM은 fold마다 학습 구간의 **정상 사진만** 학습한다. 시험은 그 fold 검증 구간의 정상 + 언더필이다.
- 판정 기준은 팀원 설정 그대로다: 정상 학습 사진 점수의 95% 지점, 점수 = 이상 지도 평균, 가장자리 5% 제외, layer3.
- 두 입력 모두 같은 원본에서 한 번 읽어 PNG(무손실)로 저장한다. 그래서 JPEG 재압축 차이가 끼어들지 않는다.

**Drive `산공종설/`에 필요한 파일**
- `underfill_originals_4000.zip`: 이미 있음
- `stfpm_v2_timeblock_mask_bundle_20261006.zip`: **새로 올릴 것** (STFPM v2 코드 사본, 실행 래퍼, 사진 목록, 마스크)

**시간 [추정]:** 패키지 설치와 입력 준비 10분 안팎, 학습은 조건마다 30분~1시간이다. 셀은 위에서부터 차례로 실행한다. 끊기면 다시 실행해도 끝난 fold는 건너뛴다.

## 1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "GPU 런타임이 아님: 런타임 > 런타임 유형 변경 > GPU"

## 2. Drive 연결 + 파일 확인

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os
DRIVE_DIR = "/content/drive/MyDrive/산공종설"
RESULTS = f"{DRIVE_DIR}/results/stfpm_v2_timeblock_mask_20261006"
os.makedirs(RESULTS, exist_ok=True)
for f in ("underfill_originals_4000.zip", "stfpm_v2_timeblock_mask_bundle_20261006.zip"):
    print("OK  " if os.path.exists(f"{DRIVE_DIR}/{f}") else "없음", f)

## 3. 압축 해제
마지막 두 줄에 원본 `4000`과 마스크 `3395`가 나와야 한다.

In [ ]:
!cp "{DRIVE_DIR}/underfill_originals_4000.zip" "{DRIVE_DIR}/stfpm_v2_timeblock_mask_bundle_20261006.zip" /content/
!unzip -q -o /content/underfill_originals_4000.zip -d /content/orig
!rm -rf /content/bundle && unzip -q -o /content/stfpm_v2_timeblock_mask_bundle_20261006.zip -d /content/bundle
!find /content/orig -name "*.jpg" | wc -l
!find /content/bundle/masks -name "*.png" | wc -l

## 4. 패키지 설치
팀원 확인 노트북과 같은 버전(anomalib 2.6.2)이다. 설치 경고는 무시해도 된다.

In [ ]:
!pip install -q "anomalib==2.6.2" timm
import importlib.metadata as md
print({p: md.version(p) for p in ("anomalib", "torch", "lightning")})
import timm
timm.create_model("resnet18", pretrained=True)   # 팀원 train.py는 HF_HUB_OFFLINE=1로 돌아서, 사전학습 가중치를 미리 받아 둔다
print("resnet18 가중치 준비 완료")

## 5. 입력 준비 (원본 PNG + 마스킹 PNG, 몇 분)
`split_b3395.csv rows per fold` 줄에 fold마다 3395가 나와야 한다.

In [ ]:
!python /content/bundle/prepare_stfpm_inputs.py --self-test
!python /content/bundle/prepare_stfpm_inputs.py --bundle /content/bundle --orig /content/orig --out /content/stfpm_inputs

## 6. 학습 + 점수: 원본 입력
각 fold의 학습 로그는 Drive `results/.../raw/foldK/train_log.txt`에 저장되고, 화면에는 기준값과 fold 결과만 나온다.

In [ ]:
COMMON = ("--project /content/bundle/STFPM_Defect_Detection --orig-root /content/stfpm_inputs --no-roi-crop "
          "--split-csv /content/stfpm_inputs/split_b3395.csv --work /content/stfpm_work")
!python /content/bundle/stfpm_timeblock.py {COMMON} --path-column raw_crop --out "{RESULTS}/raw" --train-args "--percentile 0.95"

## 7. 학습 + 점수: 마스킹 입력

In [ ]:
!python /content/bundle/stfpm_timeblock.py {COMMON} --path-column masked_crop --out "{RESULTS}/masked" --train-args "--percentile 0.95"

## 8. 결과
- **fold 평균 AUROC:** fold마다 모델이 달라 점수 크기가 다르므로, fold 안에서 AUROC를 구해 평균한다(STFPM의 기본 보고 방식).
- **같은 사진에서 원본 vs 마스킹:** 두 조건은 사진 목록·분할이 같아서, 차이는 입력(마스킹)에서만 온다.
- **2단계 통과 / 보류 활용:** 깨끗한 장면과 부분 표면 장면을 따로 본다.
- 기준 0.95의 재현율·오탐은 STFPM 자체 기준으로 판정한 결과다.

**이 셀 출력을 복사해서 Claude에게 붙여 주면 된다.** 다른 모델과 같은 사진으로 맞춘 비교는 Claude가 `scores.csv`로 계산한다.

In [ ]:
import pandas as pd, numpy as np
from sklearn.metrics import roc_auc_score
man = pd.read_csv("/content/bundle/manifest.csv", encoding="utf-8-sig").set_index("source")
def load(cond):
    d = pd.concat([pd.read_csv(f"{RESULTS}/{cond}/fold{k}/scores.csv", encoding="utf-8-sig") for k in range(5)
                   if os.path.exists(f"{RESULTS}/{cond}/fold{k}/scores.csv")])
    d["source"] = d.source.str.replace(chr(92), "/"); d["cohort"] = d.source.map(man.cohort); return d
def fold_auc(d):
    v = [roc_auc_score(g.label_id, g.score) for _, g in d.groupby("fold") if g.label_id.nunique() == 2]
    return np.mean(v), v
rows = []
for cond in ("raw", "masked"):
    if not os.path.exists(f"{RESULTS}/{cond}"): continue
    d = load(cond)
    for part, g in (("전체", d), ("2단계 통과", d[d.cohort == "primary"]), ("보류 활용(부분 표면)", d[d.cohort == "rescue"])):
        m, v = fold_auc(g); y, p = g.label_id, g.pred
        rows.append({"입력": cond, "사진": part, "n": len(g), "fold 수": d.fold.nunique(), "fold평균 AUROC": round(m, 4),
                     "fold별 AUROC": " ".join(f"{x:.3f}" for x in v),
                     "언더필 재현율(기준95)": round(((p == 1) & (y == 1)).sum() / (y == 1).sum(), 4),
                     "정상 오탐률(기준95)": round(((p == 1) & (y == 0)).sum() / (y == 0).sum(), 4)})
print(pd.DataFrame(rows).to_string(index=False))

## 9. 결과 묶기
점수·로그·설정만 Drive `results/stfpm_v2_timeblock_mask_20261006_scores.zip`으로 묶는다(체크포인트 제외). 이 zip을 Claude에게 주면 된다.

In [ ]:
import zipfile, glob
ZIP = f"{DRIVE_DIR}/results/stfpm_v2_timeblock_mask_20261006_scores.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{RESULTS}/**/*", recursive=True):
        if p.endswith((".csv", ".json", ".txt", ".yaml")):
            z.write(p, os.path.relpath(p, RESULTS))
print(ZIP, f"{os.path.getsize(ZIP) / 2**20:.1f} MiB")

## 10. 런타임 반납

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()